In [35]:
import sys
sys.path.insert(0, '../../')

In [36]:
import warnings
warnings.filterwarnings('ignore')

import sys
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score
from sklearn.base import clone
from tqdm import tqdm
from pickle import load
from scipy.stats import ttest_ind

# load local variables
from src.config import *
from src.load_models import select_model
from src.utils import calculate_r2_score, calculate_per_diff
from src.utils import calculate_y_LOD, find_score
from src.feature_selection import ModelSelection
from src.graph_visualization import feature_selection_tabularize, plot_model_vs_score, visualization_class_stratified
from src.outlier_detection import find_outliers_remove
from src.utils import evaluate_score_class_stratified

from sklearn.ensemble  import RandomForestClassifier

from sklearn.preprocessing import StandardScaler
# from sklearn.gaussian_process.kernels import Matern, RBF

In [37]:
def ValidationPerformanceWithStratified(X:pd.DataFrame,
                                        y:pd.Series,
                                        models:list,
                                        save_dir:str,
                                        kf:KFold,
                                        features_list:dict, 
                                        param_list:dict,
                                        y_LOD:float = 1.0163597575709407,
                                        use_classifier:bool=False,
                                        remove_zero:bool=False) -> None:

    
    
    for model_name in models:
        model = select_model(model_name)
            
        selected_feature_set = features_list[model_name]

        # Set the model parameters extrinsic
        model.set_params(**param_list[model_name])
    

        y_pred_all, y_true_all = [], []
        y_pred_prime_all, y_true_prime_all = [], []
        fold_test_per_error_all, fold_test_rsquare_all = [], []
        
        # Calculate score for each folds
        for train_index, test_index in kf.split(X):
            model_ = clone(model)

            # Split the data into training and testing sets
            X_train  = X.iloc[train_index]
            y_train  = y.iloc[train_index]
            
            X_test   = X.iloc[test_index]
            y_test   = y.iloc[test_index]

            # Train model with a fraction of folds only on selected features
            model_.fit(X_train[selected_feature_set], y_train)               

            
            y_pred       = model_.predict(X_test[selected_feature_set])       # prediction on remaining folds
            # y_pred_prime = model_.predict(X_test_prime[selected_feature_set]) # prediction on testing data with zero concentration removed
           
            
            # # Calculate metrics with model trained on folds
            # fold_test_per_error_all.append(calculate_per_diff(y_test_prime, y_pred_prime, y_LOD)) # Zero index data should be removed
            # fold_test_rsquare_all.append(calculate_r2_score(y_test_prime, y_pred_prime))

            y_pred_all.append(y_pred)
            y_true_all.append(y_test)

        y_pred_all = np.concatenate(y_pred_all, axis=0) 
        y_true_all = np.concatenate(y_true_all, axis=0) 

        
        # Remove zero concentration calculating
        if  remove_zero:
            y_pred_all      = pd.Series(y_pred_all)
            y_true_all      = pd.Series(y_true_all)
            
            indx_gt_zero    = list(y_true_all[y_true_all!=0].index)
            y_pred_all      = y_pred_all.loc[indx_gt_zero].to_numpy()
            y_true_all      = y_true_all.loc[indx_gt_zero].to_numpy()


        model_.fit(X[selected_feature_set], y)

        # Calculate for testing dataset
        y_pred_prime       = model_.predict(X_test_prime[selected_feature_set])
        test_rsquare_all   = calculate_r2_score(y_test_prime, y_pred_prime)
        test_per_error_all = calculate_per_diff(y_test_prime, y_pred_prime, y_LOD)

        # Calculate for all folds
        r2_score_train_folds  = calculate_r2_score(y_true_all, y_pred_all)
        per_error_train_folds = calculate_per_diff(y_true_all, y_pred_all, y_LOD)

        
        return (r2_score_train_folds, per_error_train_folds), (np.mean(test_rsquare_all), np.mean(test_per_error_all))
    

In [38]:
features_list= {'KNN':    ['min(dS/dV)', 'max(S)'], 
                'Linear': ['min(dS/dV)', 'V_max(S)', 'f2', 'f1', 'V_max(dS/dV)'], 
                'RF':     ['min(dS/dV)', 'area(S)', 'f2', 'V_max(dS/dV)'],
                'SVM':    ['min(dS/dV)', 'area(S)', 'vcenter', 'V_max(dS/dV)', 'max(dS/dV)'], 
                'GP':     ['min(dS/dV)', 'area(S)', 'V_max(dS/dV)', 'max(dS/dV)']}

param_list =  {'KNN': {'n_neighbors': 12, 'weights': 'distance', 'metric': 'manhattan'},
               'Linear': {},
               'RF': {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 2},
               'SVM': {'C': 250, 'gamma': 0.1, 'kernel': 'rbf'},
               'GP': {'kernel': 1**2 * Matern(length_scale=1, nu=1.5), 'alpha': 2.5}}

In [39]:
y_LOD = 1.0163597575709407

dataset_name = 'ML6'
remove_zero  = True
dataset_root = '../../dataset'

# Load Extracted features dataset
ML6_1 = pd.read_excel(f'{dataset_root}/ML6/Day 1 Data/feature_extraction_noise_None.xlsx')
ML6_2 = pd.read_excel(f'{dataset_root}/ML6/Day 2 Data/feature_extraction_noise_None.xlsx')

ML6_1['dataset_name'] = '1'
ML6_2['dataset_name'] = '2'
final_dataset = pd.concat([ML6_1, ML6_2], ignore_index=True)
final_dataset = pd.concat([ML6_1, ML6_2], ignore_index=True)
final_dataset['label'] = final_dataset['file'].apply(lambda x: x.split('/')[-1].split('_')[-2].replace('cbz','')).apply(lambda x: int(x))

final_dataset.rename(columns={"PH": 'max(S)', 'signal_std':'std(S)', 'signal_mean':'mean(S)', 'peak area':'area(S)', \
                        'dS_dV_area':'area(dS/dV)', 'dS_dV_max_peak':'max(dS/dV)', 'dS_dV_min_peak':'min(dS/dV)',\
                    'dS_dV_peak_diff':'max(dS/dV) - min(dS/dV)', \
                    'peak V':'V_max(S)', 'dS_dV_max_V':'V_max(dS/dV)', 'dS_dV_min_V':'V_min(dS/dV)',\
        }, inplace = True)

final_dataset = final_dataset[['area(S)', 'max(S)', 'max(dS/dV)', 'min(dS/dV)', 'V_max(S)', 'vcenter', 'V_max(dS/dV)', 'V_min(dS/dV)', 'f1', 'f2', 'file', 'label']]

max_seed      = 200
num_seed      = 100

# Assign the probability
p             = np.array([0.2]*max_seed)
p[41]         = 1.0         # 42 should be in the random state
p             = p / p.sum() # normalize the probability

random_states = np.random.choice(max_seed, size=num_seed, p=p)

# for i in range(20):
compare_test_train_r2_df         = pd.DataFrame(columns=['random_seed', 'val_score', 'test_score'])
compare_test_train_per_diff_df   = pd.DataFrame(columns=['random_seed', 'val_score', 'test_score'])

for random_state in range(1000):
    # Remove outliers from training dataset
    train, test           = train_test_split(final_dataset, test_size=0.4, shuffle=True, random_state=random_state, stratify=final_dataset['label'])
    X_filtered, outlier_S = find_outliers_remove('max(S)',  train, save_path='')
    X_filtered, outlier_S = find_outliers_remove('V_max(S)',  X_filtered, save_path='')
    
    X_train = X_filtered.drop(['label', 'file'], axis=1)
    y_train = X_filtered['label']
    X_test_prime  = test.drop(['label', 'file'], axis=1)
    y_test_prime  = test['label']
    
    if remove_zero:
        indx_gt_zero    = list(y_test_prime[y_test_prime!=0].index)
        X_test_prime         = X_test_prime.loc[indx_gt_zero]
        y_test_prime         = y_test_prime.loc[indx_gt_zero]
        
    
    models      = ['Linear']
    root_dir    = f'../../results/Journal_paper/{dataset_name}_zero_removal'
    weight_path = '../../weights/Journal_paper'
    
    kf        = KFold(n_splits=5, shuffle=True, random_state=42) 
    
    (train_fold_r2, train_fold_per_error), (test_fold_r2, test_fold_per_err) = ValidationPerformanceWithStratified(X_train, 
                                        y_train, 
                                        models, 
                                        root_dir, 
                                        kf=kf, 
                                        features_list=features_list, 
                                        param_list= param_list, 
                                        y_LOD=y_LOD,
                                        remove_zero=remove_zero,
                                        use_classifier=False)

    temp_r2     = pd.DataFrame({'random_seed':[random_state], 'val_score':[train_fold_r2], 'test_score':[test_fold_r2]})
    perError_r2 = pd.DataFrame({'random_seed':[random_state], 'val_score':[train_fold_per_error], 'test_score':[test_fold_per_err]})
    
    compare_test_train_r2_df       = pd.concat([compare_test_train_r2_df, temp_r2])
    compare_test_train_per_diff_df = pd.concat([compare_test_train_per_diff_df, perError_r2])
    

In [40]:
compare_test_train_r2_df['val_score'].mean(), compare_test_train_r2_df['test_score'].mean()

(0.7758281596173429, 0.7836786078089234)

In [41]:
compare_test_train_r2_df['diff_mean_val_score']  = compare_test_train_r2_df['val_score'].apply(lambda x: abs(x - 0.775))
compare_test_train_r2_df['diff_mean_test_score'] = compare_test_train_r2_df['test_score'].apply(lambda x: abs(x - 0.7836))
compare_test_train_r2_df['avg_diff_mean_score']  = compare_test_train_r2_df.apply(lambda x: ( x['diff_mean_val_score'] + x['diff_mean_test_score'])/2, axis=1)
sorted_output_r2  = compare_test_train_r2_df.sort_values(by=['avg_diff_mean_score'])
sorted_output_r2.head(50)

,random_seed,val_score,test_score,diff_mean_val_score,diff_mean_test_score,avg_diff_mean_score
0,622,0.774430,0.783767,0.000570,0.000167,0.000368
0,295,0.774465,0.783257,0.000535,0.000343,0.000439
0,693,0.774678,0.784233,0.000322,0.000633,0.000478
0,617,0.775615,0.784032,0.000615,0.000432,0.000523
0,941,0.775393,0.784749,0.000393,0.001149,0.000771
0,912,0.773730,0.784114,0.001270,0.000514,0.000892
0,28,0.773873,0.782739,0.001127,0.000861,0.000994
0,316,0.776854,0.783393,0.001854,0.000207,0.001031
0,75,0.775074,0.781495,0.000074,0.002105,0.001090
0,523,0.776821,0.783963,0.001821,0.000363,0.001092


In [8]:
compare_test_train_r2_df['difference'] = compare_test_train_r2_df.apply(lambda x: x['val_score'] - x['test_score'], axis=1)
train_winner = len(compare_test_train_r2_df[compare_test_train_r2_df['difference'] > 0])
total        = len(compare_test_train_r2_df)

print(train_winner/total*100)

42.9


In [9]:
compare_test_train_per_diff_df['difference'] = compare_test_train_per_diff_df.apply(lambda x: x['val_score'] - x['test_score'], axis=1)
train_winner = len(compare_test_train_per_diff_df[compare_test_train_per_diff_df['difference'] < 0])
total        = len(compare_test_train_per_diff_df)

print(train_winner/total*100)

68.2


In [10]:
compare_test_train_per_diff_df['val_score'].mean(), compare_test_train_per_diff_df['test_score'].mean()

(18.493406425781348, 19.457083815865147)

In [28]:
compare_test_train_per_diff_df['diff_mean_val_score']  = compare_test_train_per_diff_df['val_score'].apply(lambda x: abs(x - 18.49))
compare_test_train_per_diff_df['diff_mean_test_score'] = compare_test_train_per_diff_df['test_score'].apply(lambda x: abs(x - 19.45))
compare_test_train_per_diff_df['avg_diff_mean_score']  = compare_test_train_per_diff_df.apply(lambda x: ( x['diff_mean_val_score'] + x['diff_mean_test_score'])/2, axis=1)
sorted_output_per  = compare_test_train_per_diff_df.sort_values(by=['avg_diff_mean_score'])
sorted_output_per.head(50)

,random_seed,val_score,test_score,difference,diff_mean_val_score,diff_mean_test_score,avg_diff_mean_score),avg_diff_mean_score
0,579,18.510920,19.442827,-0.931906,0.020920,0.007173,0.014047,0.014047
0,559,18.444827,19.409898,-0.965072,0.045173,0.040102,0.042637,0.042637
0,122,18.587959,19.445349,-0.857390,0.097959,0.004651,0.051305,0.051305
0,963,18.409550,19.424606,-1.015056,0.080450,0.025394,0.052922,0.052922
0,330,18.517731,19.295927,-0.778196,0.027731,0.154073,0.090902,0.090902
0,651,18.520234,19.609068,-1.088834,0.030234,0.159068,0.094651,0.094651
0,564,18.411349,19.563530,-1.152181,0.078651,0.113530,0.096091,0.096091
0,526,18.456975,19.631642,-1.174667,0.033025,0.181642,0.107333,0.107333
0,157,18.280636,19.457843,-1.177207,0.209364,0.007843,0.108603,0.108603
0,431,18.411936,19.596671,-1.184736,0.078064,0.146671,0.112368,0.112368


In [34]:
sorted_output_per[sorted_output_per['random_seed']==622]
sorted_output_r2[sorted_output_r2['random_seed']==579]

,random_seed,val_score,test_score,difference,diff_mean_val_score,diff_mean_test_score,avg_diff_mean_score
0,579,0.78013,0.770708,0.009421,0.00513,0.012892,0.009011


# Paired T-test 

In [43]:
from scipy.stats import ttest_rel

# Percent Error
# Null hypothesis is both the score are the same
# Alternative per_score_train < per_score_test = 'less'
t_stat, p_value = ttest_rel(compare_test_train_per_diff_df['test_score'].tolist(),
                            compare_test_train_per_diff_df['val_score'].tolist(),
                            alternative='less')

print(p_value)

1.0


In [44]:
# R2 Score
# Null hypothesis is both the score are the same
# Alternative r2_train > r2_test = 'greater'
t_stat, p_value = ttest_rel(compare_test_train_r2_df['test_score'].tolist(),
                            compare_test_train_r2_df['val_score'].tolist(),
                            alternative='greater')

print(p_value)

1.1037171700601378e-06
